# Audio to Text - faster-whisper + Free GPU

> Model: faster-whisper large-v3 / turbo
> Platform: Google Colab (T4) or AWS Studio Lab

---


## Step 0 - Check GPU

Colab: Runtime -> Change runtime type -> **GPU (T4)**
Studio Lab: Start with **GPU** option


In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode == 0:
    print('GPU available:', r.stdout.strip())
else:
    print('No GPU detected. Please switch runtime to GPU and re-run.')


## Step 1 - Install


In [ ]:
# faster-whisper does NOT require system ffmpeg
# Uncomment next line if your audio is .m4a / .aac
# !apt-get install -y -q ffmpeg
!pip install -q faster-whisper


## Step 2 - Mount Google Drive

Put your audio file in Google Drive root, then mount here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'
print('Drive root contents:')
for f in sorted(os.listdir(DRIVE_ROOT))[:20]:
    print(' ', f)


## Step 3 - Config

| Param | Default | Notes |
|-------|---------|-------|
| MODEL_SIZE | large-v3 | Best accuracy; use turbo for speed |
| LANGUAGE | zh | Chinese; None = auto-detect |
| BATCH_SIZE | 16 | T4 has ~15GB VRAM, 16 is stable |
| VAD_FILTER | True | Skip silence to save time |
| WORD_TIMESTAMPS | True | Word-level timestamps for subtitles |


In [ ]:
# ====== Edit here ======
AUDIO_PATH  = '/content/drive/MyDrive/audio.mp3'  # your audio file
OUTPUT_PATH = '/content/drive/MyDrive/transcript.txt'
SRT_PATH    = '/content/drive/MyDrive/transcript.srt'

MODEL_SIZE      = 'large-v3'  # tiny/base/small/medium/turbo/large-v3
LANGUAGE        = 'zh'         # zh/en/ja or None for auto
BATCH_SIZE      = 16
VAD_FILTER      = True
WORD_TIMESTAMPS = True
# =======================

import os
assert os.path.exists(AUDIO_PATH), f'File not found: {AUDIO_PATH}'
print(f'Audio: {os.path.getsize(AUDIO_PATH)/1024/1024:.1f} MB')


## Step 4 - Load Model and Transcribe


In [ ]:
import time
from faster_whisper import WhisperModel, BatchedInferencePipeline

print(f'Loading model {MODEL_SIZE} ...')
t0 = time.time()
model = WhisperModel(MODEL_SIZE, device='cuda', compute_type='float16')
batched = BatchedInferencePipeline(model=model)
print(f'Model loaded in {time.time()-t0:.1f}s')

print('Transcribing...')
t1 = time.time()
segments, info = batched.transcribe(
    AUDIO_PATH,
    batch_size=BATCH_SIZE,
    vad_filter=VAD_FILTER,
    word_timestamps=WORD_TIMESTAMPS,
    language=LANGUAGE,
)
segments = list(segments)
elapsed = time.time() - t1

print(f'Done! {len(segments)} segments in {elapsed:.1f}s')
print(f'Language: {info.language} ({info.language_probability:.2%})')
print(f'Duration: {info.duration:.1f}s  realtime: {info.duration/elapsed:.1f}x')


## Step 5 - Save Output


In [ ]:
# Plain text
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for seg in segments:
        f.write(seg.text.strip() + '\n')
print(f'Text saved: {OUTPUT_PATH}')

# SRT subtitle
def fmt(sec):
    h, r = divmod(int(sec), 3600)
    m, s = divmod(r, 60)
    return f'{h:02d}:{m:02d}:{s:02d},{int((sec % 1)*1000):03d}'

with open(SRT_PATH, 'w', encoding='utf-8') as f:
    for i, seg in enumerate(segments, 1):
        f.write(f'{i}\n{fmt(seg.start)} --> {fmt(seg.end)}\n{seg.text.strip()}\n\n')
print(f'SRT saved: {SRT_PATH}')

print('\n--- Preview (first 5) ---')
for seg in segments[:5]:
    print(f'[{seg.start:.1f}s->{seg.end:.1f}s] {seg.text.strip()}')


## Step 7 — AI 优化处理（必选）

自动清理、结构化原始转录，提升可读性。

**功能**：
- 删除无意义重复、口头禅
- 补充标点，分段重组
- 提取核心主题，结构化输出
- 支持 OpenAI / DeepSeek API

**输出文件**：
- `transcript_optimized.md` — 优化后的 Markdown 文档
- `transcript_optimized.txt` — 纯文本版本


In [ ]:
# =========== AI 优化配置 ===========
USE_AI_OPTIMIZE = True  # 设为 False 可跳过
AI_MODEL = "deepseek-chat"  # "gpt-4o-mini" / "deepseek-chat"
AI_API_KEY = ""  # 在这里填入你的 API Key
MAX_TOKENS = 4000  # 处理长度限制
# ==================================

if USE_AI_OPTIMIZE and AI_API_KEY:
    print("开始 AI 优化处理...")
    
    import requests
    import json
    import time
    
    # 读取原始转录
    with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
        raw_text = f.read()
    
    # 通用优化提示词
    prompt = f"""
请将以下原始转录文本优化为结构清晰、逻辑连贯的专业文档。

【原始转录特征】
- 含口语重复、口头禅、环境噪音标注
- 句子结构松散，标点缺失
- 可能有多个说话人交替

【优化要求】
1. **清理文本**
   - 删除无意义重复（如“对对对”、“这个这个”）
   - 合并碎片短句为完整句子
   - 补充标点，分段合理

2. **结构化重组**
   - 提取核心主题作为章节标题
   - 按逻辑顺序重组内容
   - 用列表/表格整理关键数据

3. **提炼要点**
   - 提取各方观点/立场
   - 总结决策/结论
   - 标注存疑/待确认事项

4. **格式标准化**
   - 统一专有名词（如产品名、技术术语）
   - 时间戳转为参考标注（可选）
   - 输出为 Markdown

【原始文本开始】
{raw_text[:MAX_TOKENS*3]}  # 截取前部分
【原始文本结束】
"""
    
    # DeepSeek API 调用
    if "deepseek" in AI_MODEL.lower():
        url = "https://api.deepseek.com/chat/completions"
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {AI_API_KEY}"
        }
        data = {
            "model": AI_MODEL,
            "messages": [
                {"role": "system", "content": "你是一个专业的文本优化助手，擅长将口语对话转录稿转为结构化文档。"},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.3,
            "max_tokens": MAX_TOKENS
        }
        
        try:
            response = requests.post(url, headers=headers, json=data, timeout=60)
            if response.status_code == 200:
                result = response.json()["choices"][0]["message"]["content"]
            else:
                print(f"API 错误: {response.status_code}")
                result = ""
        except Exception as e:
            print(f"请求失败: {e}")
            result = ""
    
    # OpenAI API 调用（备用）
    else:
        import openai
        openai.api_key = AI_API_KEY
        try:
            response = openai.ChatCompletion.create(
                model=AI_MODEL,
                messages=[
                    {"role": "system", "content": "你是一个专业的文本优化助手，擅长将口语对话转录稿转为结构化文档。"},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,
                max_tokens=MAX_TOKENS
            )
            result = response.choices[0].message.content
        except Exception as e:
            print(f"OpenAI API 错误: {e}")
            result = ""
    
    if result:
        # 保存优化结果
        optimized_md = OUTPUT_PATH.replace('.txt', '_optimized.md')
        optimized_txt = OUTPUT_PATH.replace('.txt', '_optimized.txt')
        
        with open(optimized_md, 'w', encoding='utf-8') as f:
            f.write(result)
        with open(optimized_txt, 'w', encoding='utf-8') as f:
            f.write(result.replace('```markdown', '').replace('```', '').strip())
        
        print(f"✅ 优化完成！")
        print(f"   Markdown: {optimized_md}")
        print(f"   纯文本: {optimized_txt}")
        print("\n--- 预览（前200字符） ---")
        print(result[:200] + "...")
    else:
        print("⚠️  优化失败，请检查 API Key 或网络。")

elif USE_AI_OPTIMIZE and not AI_API_KEY:
    print("⚠️  请设置 AI_API_KEY 以启用 AI 优化功能。")
    print("   或设置 USE_AI_OPTIMIZE = False 跳过此步骤。")
else:
    print("⏭️  AI 优化已跳过。")


## Step 8 (Optional) - WhisperX Speaker Diarization

Use this to identify who is speaking.
Need: HF token from https://huggingface.co/settings/tokens
and accept license at https://huggingface.co/pyannote/speaker-diarization-3.1


In [ ]:
# Uncomment to enable speaker diarization (WhisperX)

# !pip install -q whisperx
# HF_TOKEN = 'hf_your_token_here'

# import whisperx
# audio = whisperx.load_audio(AUDIO_PATH)
# wx = whisperx.load_model('large-v3', 'cuda', compute_type='float16')
# result = wx.transcribe(audio, batch_size=16, language='zh')
# align_model, meta = whisperx.load_align_model(language_code='zh', device='cuda')
# result = whisperx.align(result['segments'], align_model, meta, audio, 'cuda')
# diarize = whisperx.DiarizationPipeline(use_auth_token=HF_TOKEN, device='cuda')
# result = whisperx.assign_word_speakers(diarize(audio), result)
# for seg in result['segments']:
#     print(f"[{seg.get('speaker','?')}] {seg['text'].strip()}")


---
## Note: Switch to AWS Studio Lab

Studio Lab keeps packages persistent. Only change the path:

```python
AUDIO_PATH = '/home/studio-lab-user/audio.mp3'
```

Everything else is identical.
